# `sparkrawl` Demo
In this notebook we demonstrate the main usage patterns that `sparkrawl` was designed for.

In [ ]:
from sparkrawl import *

# Utilities for test/demo only; not packaged.
from ospath import *
from fileio import *
from testcases import *

## (Demo Data)
`sparkrawl` is motivated by crawling custom data layouts, e.g., in Data Lakes. So for this demo we will simulate data retrieval from S3 object storage.

We start by initializing a fake data store, and populating it with example data, organized according to an ad hoc path embedding of example attributes.

In [ ]:
# Initialize fake S3.
from pathlib import Path
import tempfile

tmpdir = tempfile.TemporaryDirectory()
tmp_path = Path(tmpdir.name)

from fakes3 import *
fake_s3 = FakeS3(tmp_path / "fake-s3")

from fileio import write_jsonlines

def write_s3_json(data, uri: str):
    with tempfile.NamedTemporaryFile() as f:
        fpath = Path(f.name)
        write_jsonlines(data, fpath)
        fake_s3.put(fpath, uri)

# Populate with example data.
data_root_uri = "s3://fake-bucket/fake-prefix"
prefix = Uri.from_uri(data_root_uri)

COLORS = ["green", "red", "blue"]
YEARS = [2024, 2025]
SIZES = ["large", "small"]

import random

for c in COLORS:
    for y in YEARS:
        for sz in SIZES:
            for k in range(4):
                data = [{"x": random.random()} for _ in range(10)]
                uri_ = prefix / c / str(y) / f"size={sz}" / f"{k}.jsonl"
                uri = str(uri_)
                write_s3_json(data, uri)

def list_all(uri):
    return [
        *(uri_ for uri_ in fake_s3.list_objects(uri)),
        *(
            uri__
            for uri_ in fake_s3.list_prefixes(uri)
            for uri__ in list_all(uri_)            
        ),
    ]

print("Example data..")
print(data)
print()

print("First few partitions..")
list_all(data_root_uri)[:10]

## Branching with embedded attributes
First, we look at what we would do if we weren't using `sparkrawl`.

We see that the first branch from the root prefix encodes a color attribute. For various reasons, this field may not actually appear in the records themselves. (None do in our demo.) One common reason is to avoid storing a lot of redundant text.

For data processing purposes, however, we typical want those fields to be present in the records.
So we can define a branch iterator that yields each sub-prefix _with_ associated metadata.
In general, a branch could embed a number of attributes, so we tag with `dict`s.

In [ ]:
def list_prefixes_with_color_metadata(uri):
    """This method knows how to interpet the directory name at this level as metadata."""
    return [
        (prefix, dict(color=uri_name(prefix)))
        for prefix in fake_s3.list_prefixes(uri)
    ]
    
def uri_name(uri: str):
    return Uri.from_uri(uri).key_path.name

list_prefixes_with_color_metadata(data_root_uri)

Now we can easily query the metadata associated with any prefix to obtain its color value.

## Diving deeper
Of course, if we crawl deeper, we will want to collect all the metadata along a given path.

Let's try collecting all prefixes at depth 2.

In [ ]:
def list_prefixes_with_year_metadata(uri):
    return [
        (prefix, dict(year=int(uri_name(prefix))))
        for prefix in fake_s3.list_prefixes(uri)
    ]

def list_prefixes_with_color_and_year_metadata(tagged):
    prefix, base_attrs = tagged
    return [
        (year_prefix, {**base_attrs, **color_attrs, **year_attrs})
        for color_prefix, color_attrs in list_prefixes_with_color_metadata(prefix)
        for year_prefix, year_attrs in list_prefixes_with_year_metadata(color_prefix)
    ]

tagged = data_root_uri, {"global_attr": 42}
list_prefixes_with_color_and_year_metadata(tagged)

## Does it scale?

This approach can get out of hand quickly with many levels of nesting.
Then writing or modifying such crawlers can be time consuming and error-prone.

This is where `sparkrawl` comes in.
It lets us build crawlers out of modular components.

The central concept of `sparkrawl` is an "exploder".
An exploder is also a tagged branch iterator like those we've defined above, but it takes a _tagged_ parent element as input, rather than the parent element alone. `sparkrawl` focuses on the exploder function signature only because it is the most general function signature for the job.

In [ ]:
def as_exploder(key_iter_fn):
    
    def exploder(tagged):
        key, _attrs = tagged
        return key_iter_fn(key)

    return exploder

color_exploder = as_exploder(list_prefixes_with_color_metadata)
year_exploder = as_exploder(list_prefixes_with_year_metadata)

The central contribution of `sparkrawl` is the `explode_with` function, which decorates exploders to handle the collection of attributes during a crawl.

In [ ]:
[
    tagged__
    for tagged_ in explode_with(color_exploder)(tagged)
    for tagged__ in explode_with(year_exploder)(tagged_)
]  # same depth-2 crawl as above

`sparkrawl` provides a convenience function to simulate flatMap chains locally, also potentially useful to group stages to reduce the number of flatMaps in a RDD chain.

In [ ]:
crawler = fan_out(
    explode_with(color_exploder),
    explode_with(year_exploder),
)
list(crawler(tagged))

## Usage in RDD Spark
Everything has been done locally so far, but the same crawlers and exploders can be fed to distributed data processing frameworks like Spark.

Here's what the same crawl would look like using RDD Spark with `flatMap`.

In [ ]:
import pysparkling
sc = pysparkling.Context()

rdd = sc.parallelize([tagged])

rdd.flatMap(explode_with(color_exploder)).flatMap(explode_with(year_exploder)).collect()

## Modular crawlers

`explode_with` is un-opinionated about how you obtain your crawlers.
Users are free to implement their own modularity strategies.
However, `sparkrawl` includes a few functional programming utilities that help build exploders without writing as many custom functions.
These utilities are totally optional, but can be helpful in some instances.

Here's the same crawl with different but equivalent exploder definitions, with minimal custom functions.

In [ ]:
# Build branch-level exploders from primitives, without ad hoc function defs.
color_exploder = pipeline(key_only, fake_s3.list_prefixes, for_each(with_attribs(color=uri_name)))
year_exploder = pipeline(key_only, fake_s3.list_prefixes, for_each(with_attribs(year=pipeline(uri_name, int))))

rdd.flatMap(explode_with(color_exploder)).flatMap(explode_with(year_exploder)).collect()

We have just introduced `pipeline`, `key_only`, `for_each`, and `with_attribs`. They are just simple functional programming constructs:
- `pipeline` simply applies each function in a sequence to the output of the last one, starting with the input.
- `key_only` returns the key of a key-value pair.
- `for_each` applies its function argument to each element of an input collection.
- `with_attribs` uses a dictionary of functions to derive a dictionary of attributes to tag its input with.

We demonstrate these utilities in isolation at the end of the notebook. A few other useful functions are in the api docs.

## Iterating the data itself
So far we have only crawled to depth 2. However, now we can use the tools introduced to create a crawler for the data itself.

Below is the creation of a full data RDD in Spark.

In [ ]:
def parquet_attribs(uri):
    """
    "s3://..../size=small" -> {"size": "small"}
    """
    attrib, value = uri_name(uri).split("=", maxsplit=1)
    return {attrib: value}

parquet_exploder = pipeline(key_only, fake_s3.list_prefixes, for_each(compute_value(parquet_attribs)))
partition_exploder = pipeline(key_only, fake_s3.list_objects, for_each(with_attribs()))

def read_s3_json(uri):
    with tempfile.NamedTemporaryFile() as f:
        fake_s3.get(uri, f.name)
        yield from read_jsonlines(Path(f.name))

record_exploder = pipeline(key_only, read_s3_json, for_each(key_by_none))  # key_by_none tags None with the input dict

(rdd
    .flatMap(explode_with(color_exploder))
    .flatMap(explode_with(year_exploder))
    .flatMap(explode_with(parquet_exploder))
    .flatMap(explode_with(partition_exploder))
    .flatMap(explode_with(record_exploder))
    .map(drop_key)
).take(5)

Now each record contains all the fields obtained during the crawl to locate it.

## Reconfiguration of modular crawlers
A nice thing about the modular approach is that stages can easily be split up or recombined. For example, sometimes we wish to crawl to collect files to process in one phase, then process the data itself in another. 

Sometimes with Spark in fact, we actually collect URIs back to the main node to redistribute them more evenly again to workers.

In [ ]:
# Crawl to collect all data files.
file_crawler = fan_out(
    explode_with(color_exploder),
    explode_with(year_exploder),
    explode_with(parquet_exploder),
    explode_with(partition_exploder)
)
files_with_metadata = rdd.flatMap(file_crawler).collect()

def take(coll, n):
    return [
        i for i, _ in zip(coll, range(n))
    ]

print("Some tagged files..")
print(take(files_with_metadata, 5))
print()

# Redistribute data files and extract contents.
tagged_files_rdd = sc.parallelize(files_with_metadata)
record_rdd = tagged_files_rdd.flatMap(explode_with(record_exploder)).map(drop_key)

# Show a random sample.
count = record_rdd.count()
p = 10 / count
print("Some records..")
record_rdd.sample(withReplacement=False, fraction=p).collect()[:5]

## With Pandas
`sparkrawl` also works on `pandas.DataFrame`s, with the `explode_pandas_df` function.

Starting with the root uri in a pandas dataframe, along with global attributes..

In [ ]:
import pandas as pd

uri, attrs = tagged
records = [dict(uri=uri, **attrs)]
pdf = pd.DataFrame.from_records(records)
pdf

...we can "explode" the dataframe with the same `file_crawler` used above.

In [ ]:
file_pdf = explode_pandas_df(
    pdf,  # The source DF
    "uri",  # The name of the column to use as key
    pipeline(file_crawler, for_each(inject_key("file_uri")))  # The file crawler above with output transformed into a record iterator
)
file_pdf.head(5)

The only new utility here is `inject_key`, which inserts the key of a tagged element into its attributes dict.

In [ ]:
import copy

print("Tagged: ", tagged)
print("Injected: ", inject_key("file_uri")(copy.deepcopy(tagged)))  # `inject_key` modifies the input dict for performance reasons

## ...and Spark DataFrames too

`sparkrawl` also provides a Spark-enabled version of the same method.

In [ ]:
from pysparkling.sql.session import SparkSession
sess = SparkSession(sc)

file_sdf = sess.createDataFrame(file_pdf)

record_sdf = explode_df(
    file_sdf,
    "file_uri",
    pipeline(explode_with(record_exploder), for_each(drop_key))
)
record_sdf.show(10)

# Appendix: Functional Utility Examples

In [ ]:
# AKA https://toolz.readthedocs.io/en/latest/api.html#toolz.functoolz.compose_left
assert pipeline(range, sum, lambda x: x ** 2)(4) == sum(range(4)) ** 2

In [ ]:
tup = 5, {"ignore": "me"}
print(tup, key_only(tup))

# Note, `as_exploder` from above can be equivalently expressed in terms of `key_only`.
as_exploder_ = lambda key_iter_fn: pipeline(key_only, key_iter_fn)
color_exploder_ = as_exploder_(list_prefixes_with_color_metadata)
assert list(color_exploder_(tagged)) == list(color_exploder(tagged))

In [ ]:
with_strlen = with_attribs(dict(strlen=lambda string: len(string)))
print(with_strlen("hello"))

# and `for_each`
each_with_strlen = for_each(with_strlen)
print(list(each_with_strlen(["hello", "again!"])))